In [1]:
from dotenv import load_dotenv
import os
load_dotenv()
os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY")

In [2]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [3]:
embedding_model = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

In [4]:
from sklearn.metrics.pairwise import cosine_similarity

In [5]:
import faiss
from langchain_community.docstore import InMemoryDocstore
from langchain_community.vectorstores import FAISS

C:\Users\vansh_hxqeh4o\AppData\Local\Temp\ipykernel_16540\3224286949.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.docstore import InMemoryDocstore


In [6]:
faiss_index=faiss.IndexFlatL2(3072)#3072 is the dimension of the embedding model

In [7]:
#this is the vector store that we will use to store the embeddings and the documents
vectorstore = FAISS(
embedding_function=embedding_model, 
index=faiss_index, #this is use to store the embeddings in the FAISS index
docstore=InMemoryDocstore(),#this is used to store the documents in memory
index_to_docstore_id={},#this is used to map the index of the FAISS index to the id of the document in the docstore
)
print("Vector store created successfully")

Vector store created successfully


In [8]:
vectorstore.add_texts([
    "Washington dc is the capital of the united states",
    "Donald trump is the president of the united states",
    "Narendra modi is the prime minister of india",
])

['f6008637-a485-44ee-ad33-1d206abcae55',
 '19e734be-1f5a-4632-bb5f-1c136bff1d38',
 '657255fb-ba26-4035-a4a0-2163ec2273e6']

In [9]:
vectorstore.index_to_docstore_id

{0: 'f6008637-a485-44ee-ad33-1d206abcae55',
 1: '19e734be-1f5a-4632-bb5f-1c136bff1d38',
 2: '657255fb-ba26-4035-a4a0-2163ec2273e6'}

In [10]:
faiss_indexid=1

In [14]:
docstoreid=vectorstore.index_to_docstore_id[faiss_indexid]
print(docstoreid)

19e734be-1f5a-4632-bb5f-1c136bff1d38


In [12]:
vectorstore.docstore.search(docstoreid).page_content

'Donald trump is the president of the united states'

In [13]:
vectorstore.similarity_search("Who is the president of the united states?", k=1)

[Document(id='19e734be-1f5a-4632-bb5f-1c136bff1d38', metadata={}, page_content='Donald trump is the president of the united states')]

In [17]:
from langchain_core.documents import Document
from uuid import uuid4

doc1 = Document(
    page_content="Python is a high-level programming language widely used for AI, web development, and automation.",
    metadata={"source": "doc1", "topic": "Python"},
)

doc2 = Document(
    page_content="Machine Learning enables computers to learn patterns from data without being explicitly programmed.",
    metadata={"source": "doc2", "topic": "Machine Learning"},
)

doc3 = Document(
    page_content="Deep Learning is a subset of Machine Learning that uses neural networks with multiple layers.",
    metadata={"source": "doc3", "topic": "Deep Learning"},
)

doc4 = Document(
    page_content="Retrieval-Augmented Generation (RAG) combines information retrieval with large language models to improve response accuracy.",
    metadata={"source": "doc4", "topic": "RAG"},
)

doc5 = Document(
    page_content="FAISS is a library developed by Meta for efficient similarity search and vector indexing.",
    metadata={"source": "doc5", "topic": "FAISS"},
)

doc6 = Document(
    page_content="LangChain helps developers build applications powered by large language models using chains, agents, and retrievers.",
    metadata={"source": "doc6", "topic": "LangChain"},
)

doc7 = Document(
    page_content="Vector embeddings convert text into numerical representations that capture semantic meaning.",
    metadata={"source": "doc7", "topic": "Embeddings"},
)

doc8 = Document(
    page_content="Pandas is a Python library used for data manipulation, analysis, and cleaning.",
    metadata={"source": "doc8", "topic": "Pandas"},
)

doc9 = Document(
    page_content="NumPy provides efficient numerical computing using multidimensional arrays.",
    metadata={"source": "doc9", "topic": "NumPy"},
)

doc10 = Document(
    page_content="Computer Vision enables machines to understand and interpret images and videos.",
    metadata={"source": "doc10", "topic": "Computer Vision"},
)

documents = [
    doc1,
    doc2,
    doc3,
    doc4,
    doc5,
    doc6,
    doc7,
    doc8,
    doc9,
    doc10,
]

print("Documents created successfully")

Documents created successfully


In [18]:
vectorstore.add_documents(documents)

['1050dbe7-611d-4e4f-b3a0-1d97b453a4ac',
 'bbf4b411-3dd5-48c8-903a-a9666ebfaea5',
 '7399192d-96b9-4ec8-80be-bee35b2a2f59',
 'b06b8b77-0519-40e2-a7a5-715dc141227a',
 'e33b5d4f-f5b7-4e3e-a509-72d88a7d8e60',
 '2ec193f0-7ff8-4694-9689-6391eb0c13d4',
 '212f3018-39d5-4f88-877f-a695a015ed06',
 '7020b6a8-0ad3-4890-a6d1-1aa6e2449c2c',
 '7c850841-c73b-48c7-9639-d2b4d9e0a106',
 '2305957b-a1c2-468d-afe2-67886e992b02']

In [22]:
vectorstore.similarity_search("what is vector embeddings?", k=5,filter={"topic": "Embeddings"})

[Document(id='212f3018-39d5-4f88-877f-a695a015ed06', metadata={'source': 'doc7', 'topic': 'Embeddings'}, page_content='Vector embeddings convert text into numerical representations that capture semantic meaning.')]

In [25]:
#disk persistence of the vectorstore
vectorstore.save_local("faiss_index")

In [26]:
FAISS.load_local("faiss_index", embedding_model,allow_dangerous_deserialization=True)    

In [27]:
faiss_indexidd=vectorstore.index_to_docstore_id